In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:",torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i,torch.cuda.get_device_name(i))


CUDA available: True
GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
!pip install -q datasets transformers sentence-transformers faiss-gpu pandas numpy tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 12.8 MB/s eta 0:00:0000:0100:01


In [3]:
#libraries
import os
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import faiss
from datasets import load_dataset

In [4]:
device ="cuda" if torch.cuda.is_available() else "cpu"
gpu_count= torch.cuda.device_count()

if gpu_count>=1:
    data_size=10000
    batch_size=64
else:
    data_size=1000
    batch_size=16

print("device:", device)
print("gpu count:", gpu_count)
print("batch_size:",batch_size)
print("Data_size:",data_size)

device: cuda
gpu count: 2
batch_size: 64
Data_size: 10000


In [5]:
#Load MS marco dataset

corpus= load_dataset(
    "sentence-transformers/msmarco",
    "corpus",
    split=f"train[:{data_size}]"
)

corpus

README.md: 0.00B [00:00, ?B/s]

corpus/train-00000-of-00007.parquet:   0%|          | 0.00/238M [00:00<?, ?B/s]

corpus/train-00001-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

corpus/train-00002-of-00007.parquet:   0%|          | 0.00/242M [00:00<?, ?B/s]

corpus/train-00003-of-00007.parquet:   0%|          | 0.00/243M [00:00<?, ?B/s]

corpus/train-00004-of-00007.parquet:   0%|          | 0.00/245M [00:00<?, ?B/s]

corpus/train-00005-of-00007.parquet:   0%|          | 0.00/241M [00:00<?, ?B/s]

corpus/train-00006-of-00007.parquet:   0%|          | 0.00/240M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8841823 [00:00<?, ? examples/s]

Dataset({
    features: ['passage_id', 'passage'],
    num_rows: 10000
})

In [6]:
# convert to DataFrame and clean
df= corpus.to_pandas()

print("origional shape:",df.shape)
print(df.columns)
df=df.dropna(subset=["passage"])
df=df.drop_duplicates(subset=["passage"])
df=df.reset_index(drop=True)

print("Cleaned shape:",df.shape)
df.head()

origional shape: (10000, 2)
Index(['passage_id', 'passage'], dtype='object')
Cleaned shape: (10000, 2)


,passage_id,passage
0,0,The presence of communication amid scientific ...
1,1,The Manhattan Project and its atomic bomb help...
2,2,Essay on The Manhattan Project - The Manhattan...
3,3,The Manhattan Project was the name for a proje...
4,4,versions of each volume as well as complementa...


In [7]:
import os

os.makedirs("/kaggle/working/data",exist_ok=True)

df[["passage_id","passage"]].to_csv(
    "/kaggle/working/data/passages.csv",
    index=False
)
print("Saved:","/kaggle/working/data/passages.csv")
print("Rows:",len(df))

Saved: /kaggle/working/data/passages.csv
Rows: 10000


In [29]:
%%writefile /kaggle/working/embed_distributed.py

import os
import time
import argparse
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.distributed as dist
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, DistributedSampler

from transformers import AutoTokenizer, AutoModel


class PassageDataset(Dataset):
    def __init__(self, passages, passage_ids, tokenizer, max_length=256):
        self.passages = passages
        self.passage_ids = passage_ids
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.passages)

    def __getitem__(self, idx):
        text = str(self.passages[idx])

        encoded = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "passage_id": self.passage_ids[idx],
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0)
        }


def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    sentence_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sentence_embeddings


def setup_distributed():
    dist.init_process_group(backend="nccl")

    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])

    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    return local_rank, rank, world_size, device


def cleanup_distributed():
    dist.destroy_process_group()


def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--input_csv", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)
    parser.add_argument("--model_name", type=str, default="sentence-transformers/all-MiniLM-L6-v2")
    parser.add_argument("--batch_size", type=int, default=64)
    parser.add_argument("--max_length", type=int, default=256)

    args = parser.parse_args()

    local_rank, rank, world_size, device = setup_distributed()

    if rank == 0:
        print("Distributed embedding started")
        print("World size:", world_size)
        print("Model:", args.model_name)

    os.makedirs(args.output_dir, exist_ok=True)

    df = pd.read_csv(args.input_csv)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    model = AutoModel.from_pretrained(args.model_name)

    model.to(device)
    model.eval()

    dataset = PassageDataset(
        passages=df["passage"].tolist(),
        passage_ids=df["passage_id"].tolist(),
        tokenizer=tokenizer,
        max_length=args.max_length
    )

    sampler = DistributedSampler(
        dataset,
        num_replicas=world_size,
        rank=rank,
        shuffle=False
    )

    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        sampler=sampler,
        num_workers=2,
        pin_memory=True
    )

    local_embeddings = []
    local_passage_ids = []

    dist.barrier()
    start_time = time.time()

    with torch.no_grad():
        iterator = tqdm(loader, desc=f"Rank {rank}") if rank == 0 else loader

        for batch in iterator:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            embeddings = mean_pooling(outputs, attention_mask)
            embeddings = F.normalize(embeddings, p=2, dim=1)

            local_embeddings.append(embeddings.cpu().numpy())

            # FIX: DataLoader may convert numeric passage_id values into tensors.
            # Convert them back to normal Python values before saving to CSV.
            batch_passage_ids = batch["passage_id"]

            if torch.is_tensor(batch_passage_ids):
                batch_passage_ids = batch_passage_ids.cpu().tolist()

            local_passage_ids.extend(batch_passage_ids)

    dist.barrier()
    end_time = time.time()

    local_embeddings = np.vstack(local_embeddings).astype("float32")

    np.save(
        os.path.join(args.output_dir, f"embeddings_rank_{rank}.npy"),
        local_embeddings
    )

    pd.DataFrame({
        "passage_id": local_passage_ids
    }).to_csv(
        os.path.join(args.output_dir, f"passage_ids_rank_{rank}.csv"),
        index=False
    )

    local_count = len(local_passage_ids)
    total_time = end_time - start_time

    print(f"Rank {rank} processed {local_count} passages in {total_time:.2f} seconds")

    dist.barrier()

    if rank == 0:
        total_passages = len(df)
        throughput = total_passages / total_time

        print("=" * 80)
        print("Distributed embedding completed")
        print("Total passages:", total_passages)
        print("World size:", world_size)
        print("Total time:", round(total_time, 2), "seconds")
        print("Distributed throughput:", round(throughput, 2), "passages/sec")
        print("=" * 80)

    cleanup_distributed()


if __name__ == "__main__":
    main()

Overwriting /kaggle/working/embed_distributed.py


In [35]:
!torchrun --nproc_per_node=2 /kaggle/working/embed_distributed.py \
    --input_csv /kaggle/working/data/passages.csv \
    --output_dir /kaggle/working/distributed_outputs \
    --batch_size 64 \
    --max_length 256

W0923 07:11:14.108000 455 torch/distributed/run.py:852] 
W0923 07:11:14.108000 455 torch/distributed/run.py:852] *****************************************
W0923 07:11:14.108000 455 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0923 07:11:14.108000 455 torch/distributed/run.py:852] *****************************************
Distributed embedding started
World size: 2
Model: sentence-transformers/all-MiniLM-L6-v2
Loading weights: 100%|█| 103/103 [00:00<00:00, 1703.91it/s, Materializing param=
Loading weights:  68%|▋| 70/103 [00:00<00:00, 1248.17it/s, Materializing param=eBertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignor

In [36]:
# load and combine distributed embeddings
import os
import numpy as np
import pandas as pd

output_dir = "/kaggle/working/distributed_outputs"

embedding_files = sorted([
    f for f in os.listdir(output_dir)
    if f.lower().startswith("embeddings_rank_") and f.endswith(".npy")
])

id_files = sorted([
    f for f in os.listdir(output_dir)
    if f.lower().startswith("passage_ids_rank_") and f.endswith(".csv")
])


print("Embedding files:", embedding_files)
print("ID files", id_files)

distributed_embeddings = []
distributed_ids = []

for emb_file ,id_file in zip(embedding_files, id_files):
    emb=np.load(os.path.join(output_dir,emb_file))
    ids=pd.read_csv(os.path.join(output_dir,id_file))["passage_id"].tolist()
    distributed_embeddings.append(emb)
    distributed_ids.extend(ids)

distributed_embeddings_matrix = np.vstack(distributed_embeddings).astype("float32")

print("Distributed embedding matrix shape:", distributed_embeddings_matrix.shape)
print("Total distributed passage IDs:", len(distributed_ids))

Embedding files: ['embeddings_rank_0.npy', 'embeddings_rank_1.npy']
ID files ['passage_ids_rank_0.csv', 'passage_ids_rank_1.csv']
Distributed embedding matrix shape: (10000, 384)
Total distributed passage IDs: 10000


In [37]:
# build FAISS index from distributed embeddings
embedding_dim = distributed_embeddings_matrix.shape[1]

distributed_index = faiss.IndexFlatIP(embedding_dim)
distributed_index.add(distributed_embeddings_matrix)

print("Embedding dimension:", embedding_dim)
print("Total vectors in distributed FAISS index:", distributed_index.ntotal)

Embedding dimension: 384
Total vectors in distributed FAISS index: 10000


In [39]:
distributed_metadata = pd.DataFrame({
    "passage_id": distributed_ids
})

distributed_metadata["passage_id"] = distributed_metadata["passage_id"].astype(str)
df["passage_id"] = df["passage_id"].astype(str)

distributed_metadata = distributed_metadata.merge(
    df[["passage_id", "passage"]],
    on="passage_id",
    how="left"
)

print("Missing passages:", distributed_metadata["passage"].isna().sum())

Missing passages: 0


In [40]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

print("Tokenizer and model loaded on:", device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokenizer and model loaded on: cuda


In [41]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    sentence_embeddings = torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

    return sentence_embeddings

In [42]:
def embed_query(query):
    encoded = tokenizer(
        query,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        query_embedding = mean_pooling(outputs, attention_mask)
        query_embedding = F.normalize(query_embedding, p=2, dim=1)

    return query_embedding.cpu().numpy().astype("float32")

In [43]:
# search using distributed FAISS index
def semantic_search_distributed_index(query, top_k=5):
    query_vector = embed_query(query)

    scores, indices = distributed_index.search(query_vector, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        row = distributed_metadata.iloc[idx]

        results.append({
            "passage_id": row["passage_id"],
            "score": float(score),
            "passage": row["passage"]
        })

    return results

In [44]:
query = "what are the symptoms of diabetes"

results = semantic_search_distributed_index(query, top_k=5)

for rank, result in enumerate(results, start=1):
    print(f"Rank {rank}")
    print("Passage ID:", result["passage_id"])
    print("Score:", round(result["score"], 4))
    print("Passage:", result["passage"][:500])
    print("-" * 100)

Rank 1
Passage ID: 6052
Score: 0.5163
Passage: If you are asking what blood glucose level is considered dangerous; then, you should know that it all depends on your current health situation. However, the dangerous sugar levels, where the symptoms of hypoglycemia will occur, are considered those less than 40 mg/dL. Shaking, nausea, tremors, sweating or increased heart palpitations are some symptoms to recognize your blood sugar is dropping.
----------------------------------------------------------------------------------------------------
Rank 2
Passage ID: 6058
Score: 0.505
Passage: Random check. The doctor tests your blood sugar and itâs higher than 200, plus youâre peeing more, always thirsty, and youâve gained or lost a significant amount of weight. Heâll then do a fasting sugar level test or an oral glucose tolerance test to confirm the diagnosis.
----------------------------------------------------------------------------------------------------
Rank 3
Passage ID: 6055
Sc

In [45]:
benchmark_queries = [
    "what are the symptoms of diabetes",
    "how to improve computer performance",
    "what causes high blood pressure",
    "how does solar energy work",
    "best way to learn machine learning",
    "what is credit card fraud",
    "how to lose weight safely",
    "what is cloud computing",
    "how to treat back pain",
    "benefits of regular exercise"
]

latencies = []

for query in benchmark_queries:
    start = time.time()
    _ = semantic_search_distributed_index(query, top_k=5)
    end = time.time()

    latencies.append(end - start)

print("Total queries:", len(benchmark_queries))
print("Average distributed search latency:", round(np.mean(latencies) * 1000, 2), "ms")
print("Minimum distributed search latency:", round(np.min(latencies) * 1000, 2), "ms")
print("Maximum distributed search latency:", round(np.max(latencies) * 1000, 2), "ms")

Total queries: 10
Average distributed search latency: 8.31 ms
Minimum distributed search latency: 7.35 ms
Maximum distributed search latency: 12.8 ms
